In [1]:
##### Import general libraries

import numpy as np                      # Numerical operations
import pandas as pd                     # Data manipulation and tabular data handling
import matplotlib.pyplot as plt         # Plotting
import seaborn as sns                   # Enhanced statistical visualization

from pathlib import Path                # File system paths
import re                               # Regular expressions for parsing sample names
import warnings                         # Warning control
from collections import Counter         # Counting labels and categorical occurrences

##### Machine learning utilities

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GroupKFold,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

##### Optional notebook configuration

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")

In [2]:
# Customize plot configurations for scientific paper quality

import matplotlib as mpl
from matplotlib import rc

# Set seaborn style for aesthetics
sns.set_style("whitegrid")

# Customize matplotlib rcParams for publication quality
plt.rcParams.update({
    'font.size': 8,          # Base font size
    'axes.titlesize': 10,     # Title font size
    'axes.labelsize': 8,     # Axis label font size
    'xtick.labelsize': 8,    # X-axis tick label size
    'ytick.labelsize': 8,    # Y-axis tick label size
    'legend.fontsize': 7,    # Legend font size
    'figure.figsize': (10, 6),# Figure size (width, height)
    'figure.dpi': 300,        # Dots per inch for higher resolution
    'savefig.dpi': 300,       # Save figure DPI
    'axes.edgecolor': '0.6',  # Color of axes edges
    'axes.linewidth': 0.8,    # Width of axes edges
    'grid.color': '0.9',      # Color of grid lines
    'grid.linestyle': '--',   # Style of grid lines
    'grid.linewidth': 0.5,    # Width of grid lines
    'lines.linewidth': 1,   # Default line width
    'lines.markersize': 6,    # Default marker size
})

# Configure legend aesthetics
plt.rcParams["legend.facecolor"] = "white"
plt.rcParams["legend.edgecolor"] = '0.8'#'gray'#"black"
mpl.rcParams['legend.fontsize'] = 6
mpl.rcParams['legend.framealpha'] = 1  # Full opacity for better readability

# Enable TeX mode for high-quality text rendering
rc('font', family='serif')

In [3]:
# Install this library to enable access to the UCI Machine Learning Repository
!pip install ucimlrepo

# De los datos a la toma de decisión informada: Estudio de caso aplicado a la compresión del concreto

**¿Cómo transformar datos generados por sistemas, infraestructuras, sensores y procesos de ingeniería en información útil para la toma de decisiones?**

Actualmente muchas áreas de la ingeniería generan grandes volúmenes de datos: inspecciones de estructuras, sensores instalados en puentes o edificios, registros de mantenimiento, imágenes, datos ambientales, datos operacionales, entre otros.

Sin embargo, los datos por sí solos no son suficientes. El valor aparece cuando estos datos son organizados, procesados, interpretados y transformados en conocimiento útil.

## 1. Contexto general del estudio de caso

En este notebook se presenta un estudio de caso aplicado al análisis de datos en Ingeniería Civil, utilizando una base de datos de resistencia a la compresión del concreto.

El objetivo es construir un flujo completo de análisis que permita mostrar, de forma práctica, cómo los datos pueden transformarse en información útil para apoyar decisiones técnicas. En este caso, partiremos de datos experimentales relacionados con la composición de mezclas de concreto y su edad de curado, para analizar, modelar e interpretar la resistencia a la compresión.

La idea central del estudio de caso es:

**De los datos a la toma de decisión informada.**

Significa que no se busca solamente entrenar modelos de Machine Learning. El objetivo es desenvolver un proceso completo (_pipeline_) que incluya comprensión del problema, exploración de los datos, preparación delos mismos, modelado, evaluación, interpretación y uso de los resultados como apoyo a decisiones de ingeniería.

## 2. Descripción del problema de ingeniería

La resistencia a la compresión del concreto es una propiedad fundamental en Ingeniería Civil. Esta resistencia depende de diversos factores, incluyendo la cantidad de cemento, agua, agregados, materiales cementantes suplementarios, aditivos y el tiempo de curado.

Sin embargo, la relación entre estos factores y la resistencia final no es necesariamente lineal. Por este motivo, los métodos de Data Mining y Machine Learning pueden ser utilizados para identificar patrones, estimar propiedades mecánicas y apoyar decisiones relacionadas con dosificación, control de calidad y planificación de ensayos.

En este estudio de caso, la pregunta principal es:

> **¿Podemos estimar la resistencia a la compresión del concreto a partir de la composición de la mezcla y la edad de curado?**

A partir de esta pregunta principal, también se pueden explorar otras preguntas complementarias:

- ¿Qué variables tienen mayor relación con la resistencia del concreto?
- ¿Existen grupos de mezclas con comportamientos similares?
- ¿Es posible clasificar mezclas según niveles de resistencia?
- ¿Podemos reducir la dimensionalidad de los datos para visualizar patrones?
- ¿Qué variables parecen ser más relevantes para apoyar decisiones técnicas?

## 3. Descripción de la base de datos y possibles abordages

La base de datos utilizada corresponde a datos experimentales de resistencia a la compresión del concreto. La base contiene **1030 observaciones**, con **8 variables de entrada cuantitativas** y **1 variable de salida cuantitativa**, correspondiente a la resistencia a la compresión medida en MPa. Los datos están en forma tabular, sin valores ausentes y no escalados originalmente. :contentReference[oaicite:0]{index=0}

### 3.1 Tipo de problema

Desde el punto de vista de Machine Learning, el problema principal puede ser formulado como una tarea de **regresión supervisada**, ya que se desea predecir una variable numérica continua: la resistencia a la compresión del concreto.

En este caso, cada muestra de la base de datos está representada por un vector de variables de entrada, asociado a un valor esperado de salida. El modelo aprende, a partir de estos ejemplos, una relación aproximada entre la composición de la mezcla, la edad de curado y la resistencia medida experimentalmente.

De forma simplificada:

```text
Variables de entrada:
cemento, escoria, ceniza volante, agua, superplastificante,
agregado grueso, agregado fino, edad
        ↓
Modelo de Machine Learning
        ↓
Variable de salida:
resistencia a la compresión del concreto
```

Sin embargo, esta misma base también permite ilustrar otros enfoques importantes dentro de Data Mining y Machine Learning:

* **Regresión:** estimar la resistencia a la compresión en MPa.
* **Regresión simbólica:** estimar la resistencia en MPa a partir de una expresión matemática obtenida automáticamente desde las relaciones presentes en los datos.
* **Clasificación:** transformar la resistencia en categorías, por ejemplo baja, media y alta resistencia.
* **Clustering:** identificar grupos naturales de mezclas con características semejantes.
* **Reducción de dimensionalidad:** representar los datos en espacios de menor dimensión para facilitar la visualización de patrones.
* **Selección de características:** identificar las variables más relevantes para el desempeño e interpretación del modelo.
* **Extracción de características:** crear nuevas variables a partir de las variables originales, incorporando conocimiento de especialistas para mejorar la interpretabilidad y, eventualmente, el desempeño de los modelos.

Estos enfoques pueden ser combinados dentro de un **pipeline** coherente con la aplicación, permitiendo recorrer un flujo completo de análisis: desde la comprensión de los datos hasta la construcción de modelos e interpretación de resultados para apoyar una toma de decisión informada.


## 4. Flujo general del estudio de caso

```text

El flujo propuesto para este estudio de caso es:

Datos experimentales
        ↓
Comprensión del problema
        ↓
Análisis exploratorio de datos
        ↓
Preprocesamiento
        ↓
Representación y selección de características
        ↓
Modelos de Machine Learning
        ↓
Evaluación de resultados
        ↓
Interpretación
        ↓
Toma de decisión informada
````

Este flujo permite mostrar que un proyecto de ML aplicado a ingeniería no comienza con el algoritmo, sino con la comprensión del problema y de los datos.


## 5. Carga y organización inicial de los datos

### 5.1 Objetivo de esta etapa

El primer paso consiste en cargar la base de datos, revisar su estructura y organizar las variables de entrada y salida.

En esta etapa se busca responder preguntas simples:

* ¿Cuántas muestras tiene la base?
* ¿Cuántas variables están disponibles?
* ¿Qué tipo de datos tenemos?
* ¿Existen valores ausentes?
* ¿Cuál es la variable objetivo?
* ¿Las unidades físicas son coherentes?

### 5.2 Separación entre variables de entrada y salida

La matriz de entrada puede ser representada como:

```text
X = [cemento, escoria, ceniza_volante, agua, superplastificante,
     agregado_grueso, agregado_fino, edad]
```

La variable objetivo será:

```text
y = resistencia a la compresión del concreto
```

Esta separación es fundamental para los modelos supervisados, ya que el modelo aprende una relación aproximada entre `X` y `y`.

#### Descripción de las variables de entrada y salida

> Las **variables de entrada** representan la composición de la mezcla de concreto y la edad de curado:

| Variable | Descripción | Unidad |
|---|---|---|
| Cement | Cantidad de cemento | kg/m³ |
| Blast Furnace Slag | Cantidad de escoria de alto horno | kg/m³ |
| Fly Ash | Cantidad de ceniza volante | kg/m³ |
| Water | Cantidad de agua | kg/m³ |
| Superplasticizer | Cantidad de superplastificante | kg/m³ |
| Coarse Aggregate | Cantidad de agregado grueso | kg/m³ |
| Fine Aggregate | Cantidad de agregado fino | kg/m³ |
| Age | Edad de curado | días |

> La **variable de salida**, o variable objetivo es:

| Variable | Descripción | Unidad |
|---|---|---|
| Concrete compressive strength | Resistencia a la compresión del concreto | MPa |


In [4]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
concrete_compressive_strength = fetch_ucirepo(id=165)

# data (as pandas dataframes)
X = concrete_compressive_strength.data.features
y = concrete_compressive_strength.data.targets

# Rename variables
X.columns = ['cemento', 'escoria', 'ceniza_volante', 'agua',
             'superplastificante', 'agregado_grueso', 'agregado_fino', 'edad']

y.columns = ['resistencia'] # Original name: Concrete compressive strength

# Print input data structure
print(f"Input variables: shape{X.shape}")
display(X.head())

# Print target variable
print(f"\nTarget variable: shape{y.shape}")
display(y.head())


Input variables: shape(1030, 8)


,cemento,escoria,ceniza_volante,agua,superplastificante,agregado_grueso,agregado_fino,edad
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360



Target variable: shape(1030, 1)


,resistencia
0,79.99
1,61.89
2,40.27
3,41.05
4,44.30


# 6. Análisis exploratorio de datos

## 6.1 Objetivo del EDA

El Análisis Exploratorio de Datos, o EDA, tiene como objetivo comprender mejor la base antes de aplicar modelos. Esta etapa permite identificar distribuciones, relaciones, posibles patrones, valores extremos y asociaciones entre variables.

En este estudio de caso, el EDA será usado para conectar los datos con el conocimiento de ingeniería.

## 6.2 Análisis descriptivo

Inicialmente, se pueden calcular estadísticas básicas:

* media;
* mediana;
* desviación estándar;
* valores mínimos y máximos;
* percentiles;
* rango de variación de cada variable.

Essas estatísticas ajudam a entender se os valores observados são coerentes com o problema físico e se existem possíveis valores extremos.

## 6.3 Distribuição da resistência à compressão

Uma visualização importante é o histograma da resistência à compressão.

Essa análise permite responder:

* A maioria das amostras possui baixa, média ou alta resistência?
* A distribuição é simétrica ou assimétrica?
* Existem valores extremos?
* Há concentração de amostras em determinadas faixas de resistência?

## 6.4 Relação entre variáveis de entrada e resistência

Nesta etapa, podemos analisar gráficos de dispersão entre a resistência e algumas variáveis importantes, como:

* resistência versus idade;
* resistência versus cimento;
* resistência versus água;
* resistência versus superplastificante;
* resistência versus escória;
* resistência versus cinza volante.

Essas visualizações permitem discutir relações esperadas do ponto de vista da engenharia, por exemplo:

* aumento da idade tende a aumentar a resistência;
* maior quantidade de cimento pode estar associada a maior resistência;
* excesso de água pode reduzir a resistência;
* adições minerais podem modificar o ganho de resistência ao longo do tempo.

## 6.5 Matriz de correlação

A matriz de correlação pode ser usada para observar associações lineares entre variáveis.

No entanto, é importante destacar que:

> Correlação não implica causalidade, e relações não lineares podem não ser bem representadas por uma correlação linear simples.

Mesmo assim, essa análise é útil para uma primeira aproximação e para identificar possíveis redundâncias entre variáveis.

---

# 7. Preprocesamiento de los datos

## 7.1 Objetivo del preprocesamiento

El preprocesamiento tiene como objetivo preparar los datos para que puedan ser utilizados adecuadamente por los modelos de Machine Learning.

Aunque esta base es relativamente limpia, todavía es importante realizar algunas etapas básicas:

* verificar valores ausentes;
* verificar tipos de datos;
* separar entrada y salida;
* dividir datos en entrenamiento y prueba;
* escalar variables cuando sea necesario.

## 7.2 Verificación de valores ausentes

Según la descripción de la base, no existen valores ausentes. Aun así, en un flujo real de análisis siempre es recomendable verificar esta condición antes de entrenar modelos.

## 7.3 División en entrenamiento y prueba

Para evaluar la capacidad de generalización del modelo, los datos deben separarse en dos conjuntos:

* **Conjunto de entrenamiento:** usado para ajustar el modelo.
* **Conjunto de prueba:** usado para evaluar el desempeño en datos no vistos.

Una división típica puede ser:

```text
80% entrenamiento
20% prueba
```

También se puede usar validación cruzada para obtener una estimación más robusta del desempeño.

## 7.4 Escalamiento de variables

Algunos modelos son sensibles a la escala de las variables. Por ejemplo:

* Regresión lineal regularizada;
* Support Vector Machine;
* K-Nearest Neighbors;
* Redes neuronales;
* PCA.

Por eso, puede ser necesario aplicar:

* normalización;
* estandarización;
* escalamiento robusto.

Modelos basados en árboles, como Decision Tree y Random Forest, normalmente son menos sensibles a la escala de las variables.

---

# 8. Ingeniería y selección de características

## 8.1 Representación de los datos

Una parte central del flujo de Data Mining y ML es la representación de los datos.

Embora a base já forneça variáveis diretamente relacionadas à composição da mistura, podemos criar novas variáveis derivadas com significado físico ou prático.

## 8.2 Possíveis variáveis derivadas

Algumas variáveis derivadas interessantes são:

### Relação água/cimento

```text
water_cement_ratio = water / cement
```

Essa variável é importante porque a relação água/cimento está fortemente associada à resistência e à durabilidade do concreto.

### Total de materiais cimentícios

```text
binder = cement + blast_furnace_slag + fly_ash
```

Essa variável representa a soma dos principais materiais cimentícios.

### Relação água/material cimentício

```text
water_binder_ratio = water / binder
```

Essa variável pode ser mais informativa do que a relação água/cimento quando existem adições minerais.

### Proporção de adições minerais

```text
slag_ratio = blast_furnace_slag / binder
fly_ash_ratio = fly_ash / binder
```

Essas variáveis podem ajudar a interpretar o papel da escória e da cinza volante na mistura.

## 8.3 Seleção de características

A seleção de características busca identificar quais variáveis são mais relevantes para explicar ou prever a resistência.

Podem ser utilizadas diferentes estratégias:

* análise de correlação;
* importância de variáveis em modelos baseados em árvores;
* eliminação recursiva de variáveis;
* métodos baseados em regularização;
* comparação de desempenho com diferentes subconjuntos de variáveis.

O objetivo não é apenas melhorar o modelo, mas também apoiar a interpretação:

> Quais fatores parecem mais importantes para estimar a resistência à compressão?

---

# 9. Redução de dimensionalidade

## 9.1 Objetivo

A redução de dimensionalidade permite representar os dados em um espaço menor, geralmente com duas ou três dimensões, facilitando a visualização de padrões.

Neste estudo de caso, podemos usar redução de dimensionalidade para observar se as misturas formam grupos naturais ou tendências associadas à resistência.

## 9.2 PCA

A Análise de Componentes Principais, ou PCA, pode ser usada para projetar os dados em duas componentes principais.

Possíveis objetivos:

* visualizar a estrutura geral dos dados;
* observar agrupamentos;
* identificar tendências relacionadas à resistência;
* detectar possíveis amostras atípicas;
* avaliar redundância entre variáveis.

## 9.3 Interpretação

Após aplicar PCA, podemos colorir os pontos de acordo com:

* valor da resistência;
* faixa de resistência;
* idade de cura;
* relação água/cimento;
* grupos encontrados por clustering.

Isso permite conectar visualização, engenharia e interpretação dos modelos.

---

# 10. Clustering: identificação de grupos de misturas

## 10.1 Objetivo

O clustering é uma técnica não supervisionada usada para identificar grupos naturais nos dados, sem utilizar previamente a variável de saída.

Neste estudo de caso, o clustering pode responder:

* Existem grupos de misturas com composições semelhantes?
* Esses grupos apresentam resistências diferentes?
* Algumas famílias de misturas podem ser associadas a maior ou menor desempenho?
* O agrupamento revela padrões úteis para análise de engenharia?

## 10.2 Métodos possíveis

Podemos aplicar métodos como:

* K-Means;
* DBSCAN;
* clustering hierárquico.

Para uma apresentação introdutória, K-Means pode ser usado como método principal por ser simples e intuitivo.

## 10.3 Interpretação dos clusters

Após encontrar os grupos, podemos analisar:

* média da resistência em cada cluster;
* composição média de cada grupo;
* idade média de cura;
* relação água/cimento média;
* distribuição das amostras por grupo.

A interpretação dos clusters pode ajudar a transformar uma técnica matemática em uma análise útil para engenharia.

---

# 11. Regressão: previsão da resistência à compressão

## 11.1 Objetivo

A principal tarefa de Machine Learning com esta base é a regressão.

O objetivo é estimar a resistência à compressão do concreto a partir das variáveis de entrada.

```text
Composição da mistura + idade
        ↓
Modelo de regressão
        ↓
Resistência estimada em MPa
```

## 11.2 Modelos sugeridos

Para fins didáticos, podemos comparar modelos com diferentes níveis de complexidade:

| Modelo              | Papel no estudo                           |
| ------------------- | ----------------------------------------- |
| Regressão Linear    | Modelo simples de referência              |
| Árvore de Decisão   | Modelo interpretável                      |
| Random Forest       | Modelo robusto para relações não lineares |
| Gradient Boosting   | Modelo mais avançado para comparação      |
| Rede Neural simples | Conexão com IA e aprendizado não linear   |

## 11.3 Métricas de avaliação

As principais métricas para regressão são:

### MAE — Mean Absolute Error

Mede o erro médio absoluto entre valores reais e previstos.

Interpretação simples:

> Em média, quantos MPa o modelo erra?

### RMSE — Root Mean Squared Error

Penaliza erros maiores de forma mais intensa.

Interpretação simples:

> O modelo está cometendo grandes erros em algumas amostras?

### R² — Coeficiente de determinação

Mede a proporção da variação da resistência que é explicada pelo modelo.

Interpretação simples:

> Quanto da variação dos dados o modelo consegue explicar?

## 11.4 Visualização dos resultados

Visualizações recomendadas:

* gráfico de valores reais versus valores previstos;
* distribuição dos erros;
* gráfico de resíduos;
* comparação entre modelos;
* importância das variáveis.

Essas visualizações ajudam a discutir não apenas se o modelo “funciona”, mas também onde ele erra e por quê.

---

# 12. Classificação: transformação da resistência em categorias

## 12.1 Objetivo

Embora o problema original seja de regressão, também podemos transformar a resistência em classes para construir um exemplo de classificação.

Por exemplo, a resistência pode ser dividida em faixas:

```text
Baixa resistência
Média resistência
Alta resistência
```

Essa transformação permite discutir outro tipo importante de problema de Machine Learning:

> Em vez de prever o valor exato da resistência, o modelo pode classificar a mistura em uma categoria de desempenho.

## 12.2 Criação das classes

As classes podem ser definidas de diferentes formas:

### Por critérios estatísticos

Usando tercis ou quartis da própria distribuição:

```text
Baixa: valores abaixo do primeiro tercil
Média: valores entre o primeiro e o segundo tercil
Alta: valores acima do segundo tercil
```

### Por critérios de engenharia

Usando limites técnicos definidos com base em normas, especificações ou objetivos do projeto.

Para fins didáticos, a divisão por tercis pode ser suficiente. Para aplicações reais, limites técnicos devem ser definidos com cuidado.

## 12.3 Modelos de classificação

Modelos possíveis:

* Regressão logística;
* Árvore de decisão;
* Random Forest;
* Support Vector Machine;
* K-Nearest Neighbors.

## 12.4 Métricas de avaliação

As principais métricas para classificação são:

* acurácia;
* matriz de confusão;
* precisão;
* recall;
* F1-score.

A matriz de confusão é especialmente didática porque mostra onde o modelo acerta e onde confunde categorias.

## 12.5 Interpretação prática

A classificação pode apoiar decisões como:

* identificar misturas com maior probabilidade de baixa resistência;
* priorizar ensaios adicionais;
* apoiar controle de qualidade;
* separar misturas em níveis de desempenho.

---

# 13. Detecção de anomalias e amostras atípicas

## 13.1 Objetivo

Além de regressão, classificação e clustering, também podemos investigar possíveis amostras atípicas.

A detecção de anomalias pode ser útil para identificar:

* misturas incomuns;
* valores extremos;
* possíveis erros de medição;
* combinações raras de componentes;
* amostras com resistência inesperada.

## 13.2 Estratégias possíveis

Algumas estratégias simples incluem:

* análise de boxplots;
* distância em relação à média;
* análise no espaço PCA;
* Isolation Forest;
* comparação entre erro de previsão e composição da mistura.

## 13.3 Interpretação

Uma amostra pode ser considerada interessante quando:

* possui composição incomum;
* apresenta resistência muito maior ou menor que o esperado;
* gera erro elevado no modelo;
* pertence a uma região pouco representada dos dados.

Esses casos podem ser discutidos como oportunidades para investigação técnica.

---

# 14. Interpretação dos modelos

## 14.1 Importância da interpretação

Em aplicações de engenharia, não basta obter uma boa métrica. Também é necessário entender o comportamento do modelo.

A interpretação permite responder:

* Quais variáveis mais influenciam a previsão?
* O modelo está coerente com o conhecimento físico?
* Existem relações inesperadas?
* O modelo pode ser usado como apoio à decisão?
* Quais são suas limitações?

## 14.2 Importância das variáveis

Modelos como Random Forest permitem estimar a importância relativa das variáveis.

Variáveis esperadas como importantes podem incluir:

* idade;
* cimento;
* água;
* relação água/cimento;
* superplastificante;
* materiais cimentícios suplementares.

## 14.3 Relação com conhecimento de engenharia

A interpretação deve sempre ser conectada ao domínio físico.

Por exemplo:

* Se a idade aparece como variável importante, isso é coerente com o processo de cura.
* Se a água aparece com influência negativa, isso pode estar relacionado à relação água/cimento.
* Se materiais suplementares influenciam o resultado, isso pode indicar efeitos de composição e reações ao longo do tempo.

---

# 15. Da previsão à tomada de decisão

## 15.1 Transformando resultados em apoio à decisão

O objetivo final do estudo de caso é mostrar como os resultados dos modelos podem apoiar decisões técnicas.

O fluxo pode ser resumido assim:

```text
Dados da mistura
        ↓
Modelo treinado
        ↓
Resistência estimada
        ↓
Interpretação dos fatores relevantes
        ↓
Apoio à decisão
```

## 15.2 Exemplos de decisões possíveis

Com base nos resultados, um engenheiro poderia:

* estimar a resistência esperada antes de realizar todos os ensaios;
* comparar diferentes formulações;
* identificar misturas promissoras;
* avaliar se uma mistura atende a um requisito mínimo;
* reduzir tentativa e erro em laboratório;
* priorizar ensaios experimentais;
* apoiar controle de qualidade;
* identificar combinações que merecem investigação adicional.

## 15.3 Limitações importantes

É importante destacar que o modelo não substitui ensaios laboratoriais, normas técnicas ou julgamento profissional.

O modelo deve ser entendido como uma ferramenta de apoio, não como decisão automática.

Algumas limitações incluem:

* a base representa um conjunto específico de misturas;
* o modelo aprende a partir dos dados disponíveis;
* extrapolações para misturas muito diferentes podem ser arriscadas;
* a qualidade da decisão depende da qualidade dos dados;
* resultados devem ser interpretados com conhecimento de engenharia.

---

# 16. Síntese do pipeline completo

Ao final do estudo de caso, o pipeline pode ser apresentado da seguinte forma:

```text
1. Entender o problema
   ↓
2. Conhecer os dados
   ↓
3. Explorar padrões iniciais
   ↓
4. Preparar e transformar os dados
   ↓
5. Criar ou selecionar características relevantes
   ↓
6. Aplicar redução de dimensionalidade para visualização
   ↓
7. Identificar grupos com clustering
   ↓
8. Treinar modelos de regressão
   ↓
9. Construir uma versão de classificação
   ↓
10. Avaliar os modelos
   ↓
11. Interpretar os resultados
   ↓
12. Apoiar a tomada de decisão informada
```

---

# 17. Mensagem principal do estudo de caso

Este estudo de caso mostra que Machine Learning, Data Mining e Inteligência Artificial podem ser integrados a problemas reais de Engenharia Civil de maneira clara, prática e interpretável.

A principal mensagem é:

> O valor da IA em engenharia não está apenas no algoritmo, mas na capacidade de transformar dados em informação, informação em conhecimento e conhecimento em decisões técnicas melhor fundamentadas.

Neste sentido, a base de resistência à compressão do concreto permite demonstrar, de forma concreta, todo o percurso:

```text
Dados → Informação → Modelo → Interpretação → Decisão
```